# Setup

Add libraries and configure with Drive

In [1]:
from google.colab import drive
import pandas as pd
import xml.etree.ElementTree as ET
import os

drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


# PLABA Data Extraction and Cleanup

In [2]:
import zipfile
import pandas as pd

zip_path = '/content/drive/MyDrive/Google 2F - FaithfulMed/FaithfulMed Data Simplifier/PLABA.zip'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('/content/plaba_unzipped')

train_df = pd.read_csv('/content/plaba_unzipped/train.csv')
val_df = pd.read_csv('/content/plaba_unzipped/val.csv')
test_df = pd.read_csv('/content/plaba_unzipped/test.csv')

plaba_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

plaba_clean = plaba_df[['input_text', 'target_text']].copy()
plaba_clean.columns = ['Medical_Term', 'Lay_Language_Definiton']

plaba_clean['Metadata'] = plaba_df.apply(
    lambda row: {
        'source': 'PLABA',
        'pmid': str(row['pmid']),
        'question': row['question'],
        'adaptation_version': row['Adaptation_Version'],
        'question_type': row['Question_Type']
    }, axis=1
)

plaba_clean.head()

,Medical_Term,Lay_Language_Definiton,Metadata
0,Exercise-Associated Muscle Cramps (EAMC) are a...,Exercise-Associated Muscle Cramps (EAMC) are a...,"{'source': 'PLABA', 'pmid': '29857264', 'quest..."
1,"Background: Muscle cramp is a painful, involun...",Muscle cramps are unconscious contractions of ...,"{'source': 'PLABA', 'pmid': '33722257', 'quest..."
2,Muscle cramp is a temporary but intense and pa...,"Muscle cramp is a temporary but intense, painf...","{'source': 'PLABA', 'pmid': '31696455', 'quest..."
3,Muscular cramp is a common symptom in healthy ...,"Muscle cramps are common in healthy people, es...","{'source': 'PLABA', 'pmid': '30168894', 'quest..."
4,"Muscle cramps result in continuous, involuntar...",Muscle cramps cause constant and unintended co...,"{'source': 'PLABA', 'pmid': '29763070', 'quest..."


# MedLine Plus Glossary Extraction and Cleanup

In [3]:
import xml.etree.ElementTree as ET
import pandas as pd
import zipfile
import os

medline_zip_path = '/content/drive/MyDrive/Google 2F - FaithfulMed/FaithfulMed Data Simplifier/MedLineGlossary.zip'
extract_folder = '/content/medline_unzipped'

with zipfile.ZipFile(medline_zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_folder)

xml_file_path = None
for root_dir, dirs, files in os.walk(extract_folder):
    for file in files:
        if file.endswith('.xml'):
            xml_file_path = os.path.join(root_dir, file)
            break
    if xml_file_path:
        break

if xml_file_path:
    tree = ET.parse(xml_file_path)
    root = tree.getroot()

    for elem in root.iter():
        if '}' in elem.tag:
            elem.tag = elem.tag.split('}', 1)[1]

    medline_records = []

    for topic in root.findall('.//health-topic'):
        # ONLY process English topics - we do not need spanish for now
        if topic.attrib.get('language') != 'English':
            continue

        term_title = topic.attrib.get('title', 'Unknown Term')
        summary_element = topic.find('.//full-summary')

        if summary_element is not None:
            summary_text = "".join(summary_element.itertext()).strip()

            if summary_text:
                medline_records.append({
                    'Medical_Term': term_title,
                    'Lay_Language_Definition': summary_text,
                    'Metadata': {
                        'source': 'MedlinePlus',
                        'url': topic.attrib.get('url', 'No URL')
                    }
                })

    medline_df = pd.DataFrame(medline_records)
    master_df = pd.concat([plaba_clean, medline_df], ignore_index=True)

    print(f"Added {len(medline_df)} English MedlinePlus records.")
    print(f"Total dictionary size so far: {len(master_df)}")

Added 1015 English MedlinePlus records.
Total dictionary size so far: 1936


# MedQuAd Extraction and Clean

In [5]:
import os
import xml.etree.ElementTree as ET
import pandas as pd
import zipfile

# 1. Paths
medquad_zip_path = '/content/drive/MyDrive/Google 2F - FaithfulMed/FaithfulMed Data Simplifier/MedQuAD.zip'
extract_folder = '/content/medquad_unzipped'
medquad_base = '/content/medquad_unzipped/MedQuAD-master'

# 2. Unzip the file fresh
print("Unzipping MedQuAD from Google Drive...")
with zipfile.ZipFile(medquad_zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_folder)

# 3. Sweep for data
medquad_records = []
xml_files_found = 0
qa_pairs_extracted = 0

print("Scanning MedQuAD folders for Q&A pairs...")
for root_dir, dirs, files in os.walk(medquad_base):
    for file in files:
        if file.endswith('.xml'):
            xml_files_found += 1
            file_path = os.path.join(root_dir, file)

            try:
                tree = ET.parse(file_path)
                root = tree.getroot()

                focus_node = root.find('.//Focus')
                focus = focus_node.text if focus_node is not None else "Unknown"

                for qa in root.findall('.//QAPair'):
                    question = qa.find('Question')
                    answer = qa.find('Answer')

                    if question is not None and answer is not None and answer.text:
                        qa_pairs_extracted += 1
                        medquad_records.append({
                            'Medical_Term': question.text.strip(),
                            'Lay_Language_Definition': answer.text.strip(),
                            'Metadata': {
                                'source': 'MedQuAD',
                                'topic_focus': focus
                            }
                        })
            except Exception as e:
                continue

print(f"Diagnostics: Found {xml_files_found} XML files.")
print(f"Diagnostics: Extracted {qa_pairs_extracted} Q&A pairs.")

# 4. Merge with the master dataframe
if qa_pairs_extracted > 0:
    medquad_df = pd.DataFrame(medquad_records)
    final_master_df = pd.concat([master_df, medquad_df], ignore_index=True)
    print(f"\nSUCCESS! Final dictionary size ready for embeddings: {len(final_master_df)}")
else:
    print("\nFAILURE: No Q&A pairs found.")

Unzipping MedQuAD from Google Drive...
Scanning MedQuAD folders for Q&A pairs...
Diagnostics: Found 11274 XML files.
Diagnostics: Extracted 16407 Q&A pairs.

SUCCESS! Final dictionary size ready for embeddings: 18343


# Embedding

In [13]:
# Install local embedding library
!pip install -q -U sentence-transformers

from sentence_transformers import SentenceTransformer
import pandas as pd
import json

# Load standard local model
print("Downloading open-source embedding model to Colab...")
model = SentenceTransformer('all-MiniLM-L6-v2')

# output path
output_file = '/content/drive/MyDrive/Google 2F - FaithfulMed/FaithfulMed Data Simplifier/RAG_Embeddings_Full.jsonl'

print(f"Starting local processing. Total rows: {len(final_master_df)}")

# Generate embeddings locally
text_list = final_master_df['Text_to_Embed'].tolist()

print("Generating embeddings... (This will take ~2 minutes)")
# batching in the background
embeddings = model.encode(text_list, show_progress_bar=True)

# Attach vectors back
final_master_df['Embedding'] = embeddings.tolist()

# Save
print("Saving dataset...")
with open(output_file, 'w') as f:
    for _, row in final_master_df.iterrows():
        f.write(row.to_json() + '\n')

print(f"\nSUCCESS! File safely saved to {output_file}.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 13.1 MB/s eta 0:00:00


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Starting local processing. Total rows: 18343
Generating embeddings... (This will take ~2 minutes)


Batches:   0%|          | 0/574 [00:00<?, ?it/s]

Saving dataset...

SUCCESS! File safely saved to /content/drive/MyDrive/Google 2F - FaithfulMed/FaithfulMed Data Simplifier/RAG_Embeddings_Full.jsonl.


In [ ]:
import